# Chapter 19: Structure of SLAM

<a href="../lite/lab/index.html?path=ch19_structure_of_slam.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

SLAM is a chicken and egg problem. To build a map, you need to know where you are.
To know where you are, you need a map. Solving both at once sounds impossible. But there
is a beautiful mathematical structure hiding in this problem, and once you see it, SLAM
becomes not just possible, but elegant.

This chapter reveals that structure: the **joint estimation** of robot poses and landmark
positions, and the **correlations** that make the whole system work.

```{admonition} What you will build
:class: tip

- Construct the joint SLAM state vector containing both robot poses and landmark positions
- Watch correlations grow in the covariance matrix as the robot observes landmarks
- See how loop closure dramatically reduces uncertainty across the entire trajectory
- Understand why SLAM is fundamentally about managing and exploiting correlations

**Real world application:** This chapter reveals the mathematical structure that makes SLAM possible. Understanding joint estimation and correlations is the key insight behind every SLAM algorithm.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **GTSAM** | Georgia Tech's factor graph library, the gold standard for SLAM research |
| **g2o** | General graph optimization, used in ORB-SLAM, RTAB-Map |
| **Ceres Solver** | Google's optimizer, used in Cartographer and many SLAM systems |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 19.1 Joint Estimation

In SLAM, the state vector contains **both** the robot pose and all landmark positions:

$$\mathbf{x} = \begin{bmatrix} x_R \\ y_R \\ \theta_R \\ l_{1x} \\ l_{1y} \\ \vdots \\ l_{Nx} \\ l_{Ny} \end{bmatrix}$$

The covariance matrix captures the uncertainty of each variable **and** the correlations between them.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_landmarks = 4
robot_pos = np.array([5.0, 5.0])
landmark_positions = np.array([[2, 3], [8, 2], [7, 8], [3, 7]])[:n_landmarks]
# ──────────────────────────────────────────────────────────────────────────────

state_dim = 2 + 2 * n_landmarks  # robot (x,y) + landmarks (x,y each)

# Build a simple covariance: robot uncertain, landmarks more uncertain, no correlations yet
P = np.eye(state_dim)
P[0, 0] = P[1, 1] = 0.5  # robot uncertainty
for i in range(n_landmarks):
    P[2+2*i, 2+2*i] = 2.0  # landmark x uncertainty
    P[2+2*i+1, 2+2*i+1] = 2.0  # landmark y uncertainty

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: the SLAM state
ax = axes[0]
ax.plot(*robot_pos, 'ko', ms=12, zorder=5)
ax.annotate('Robot', xy=robot_pos, xytext=(5, -15), textcoords='offset points', fontsize=11, fontweight='bold')
draw_cov_ellipse(ax, robot_pos, P[:2, :2], fill=True, facecolor='steelblue', alpha=0.2, edgecolor='steelblue', lw=2)

for i, lm in enumerate(landmark_positions):
    ax.scatter(*lm, c='tomato', s=100, marker='^', zorder=5)
    ax.annotate(f'L{i}', xy=lm, xytext=(5, 5), textcoords='offset points', fontsize=10, color='tomato')
    cov_lm = P[2+2*i:2+2*i+2, 2+2*i:2+2*i+2]
    draw_cov_ellipse(ax, lm, cov_lm, fill=True, facecolor='tomato', alpha=0.15, edgecolor='tomato', lw=1.5)

ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect('equal')
ax.set_title("SLAM state: robot + landmarks with uncertainty", fontsize=13)

# Right: covariance matrix
ax = axes[1]
im = ax.imshow(np.abs(P), cmap='Blues', interpolation='nearest')
ax.set_title("Covariance matrix P (no correlations yet)", fontsize=13)
ax.set_xlabel("State index"); ax.set_ylabel("State index")
plt.colorbar(im, ax=ax, shrink=0.8)

labels = ['Rx', 'Ry'] + [f'L{i}{c}' for i in range(n_landmarks) for c in ['x','y']]
ax.set_xticks(range(state_dim)); ax.set_xticklabels(labels, fontsize=8, rotation=45)
ax.set_yticks(range(state_dim)); ax.set_yticklabels(labels, fontsize=8)

plt.tight_layout()
plt.show()

## 19.2 Correlation

Here is the magic of SLAM: when the robot **observes a landmark**, it creates a
**correlation** between the robot pose and that landmark in the covariance matrix.
Over time, landmarks become correlated with each other too, because they are all
linked through the robot.

This means that improving the estimate of **one** landmark can improve the estimates
of **all** landmarks. The entire map learns from every single observation.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_observations = 15     # number of observations to simulate  (try 1, 5, 15, 30)
sigma_obs = 0.5         # observation noise
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
P_evolving = np.eye(state_dim) * 2.0
P_evolving[0,0] = P_evolving[1,1] = 0.3  # robot starts with some certainty

# Simulate observations: robot sees random landmarks
for obs in range(n_observations):
    lm_idx = np.random.randint(n_landmarks)
    # Observation Jacobian: H maps state to measurement
    H = np.zeros((2, state_dim))
    H[0, 0] = -1; H[0, 2+2*lm_idx] = 1    # z_x = lm_x - robot_x
    H[1, 1] = -1; H[1, 2+2*lm_idx+1] = 1  # z_y = lm_y - robot_y
    R_obs = np.eye(2) * sigma_obs**2
    
    # Kalman update
    S = H @ P_evolving @ H.T + R_obs
    K = P_evolving @ H.T @ np.linalg.inv(S)
    P_evolving = (np.eye(state_dim) - K @ H) @ P_evolving

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(*robot_pos, 'ko', ms=12, zorder=5)
draw_cov_ellipse(ax, robot_pos, P_evolving[:2, :2], fill=True, facecolor='steelblue', alpha=0.2, edgecolor='steelblue', lw=2)
for i, lm in enumerate(landmark_positions):
    ax.scatter(*lm, c='tomato', s=100, marker='^', zorder=5)
    cov_lm = P_evolving[2+2*i:2+2*i+2, 2+2*i:2+2*i+2]
    draw_cov_ellipse(ax, lm, cov_lm, fill=True, facecolor='tomato', alpha=0.15, edgecolor='tomato', lw=1.5)
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect('equal')
ax.set_title(f"After {n_observations} observations: smaller ellipses!", fontsize=13)

ax = axes[1]
im = ax.imshow(np.abs(P_evolving), cmap='Blues', interpolation='nearest')
ax.set_title(f"Covariance after {n_observations} obs (note the correlations!)", fontsize=13)
labels = ['Rx', 'Ry'] + [f'L{i}{c}' for i in range(n_landmarks) for c in ['x','y']]
ax.set_xticks(range(state_dim)); ax.set_xticklabels(labels, fontsize=8, rotation=45)
ax.set_yticks(range(state_dim)); ax.set_yticklabels(labels, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

print("Off-diagonal blocks are non-zero = correlations between robot and landmarks!")
print(f"Correlation between Robot and L0: {P_evolving[0, 2]:.4f}")

## 19.3 Drift

Without **loop closure**, the robot's pose uncertainty grows over time due to odometry noise.
Landmarks observed early become well estimated, but landmarks observed later inherit the
robot's accumulated drift.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_steps = 30
sigma_motion = 0.2     # motion noise per step
sigma_obs = 0.3        # observation noise
# ──────────────────────────────────────────────────────────────────────────────

# Simulate robot driving in a line, observing landmarks along the way
poses = np.zeros((n_steps, 2))
for i in range(1, n_steps):
    poses[i] = poses[i-1] + np.array([1.0, 0.0])  # drive right

# Track position uncertainty growth
position_sigmas = [0.1]  # initial
for i in range(1, n_steps):
    position_sigmas.append(np.sqrt(position_sigmas[-1]**2 + sigma_motion**2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(poses[:, 0], poses[:, 1], 'steelblue', lw=2, marker='o', ms=4)
for i in range(0, n_steps, 3):
    cov = np.eye(2) * position_sigmas[i]**2
    draw_cov_ellipse(ax, poses[i], cov, fill=True, facecolor='steelblue', alpha=0.1, edgecolor='steelblue', lw=1)
ax.set_title("Uncertainty grows with each step (no measurements)", fontsize=13)
ax.set_aspect('equal')

ax = axes[1]
ax.plot(position_sigmas, 'tomato', lw=2, marker='o', ms=4)
ax.set_xlabel("Time step"); ax.set_ylabel("Position σ (m)")
ax.set_title("Uncertainty growth: $\\sigma_t = \\sqrt{\\sigma_{t-1}^2 + \\sigma_{motion}^2}$", fontsize=13)

plt.tight_layout()
plt.show()

## 19.4 Loop Closure

**Loop closure** is the moment the robot recognizes a previously visited place.
This single event dramatically reduces uncertainty, not just for the current pose,
but for the **entire trajectory** and map. It is the most powerful event in SLAM.

When the robot revisits pose 0 from pose 30, the correlation propagates backward,
correcting every intermediate pose.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_poses = 20
sigma_odom = 0.3
sigma_loop = 0.1       # loop closure observation noise
show_loop_closure = True   # toggle loop closure on/off (try True vs False)
# ──────────────────────────────────────────────────────────────────────────────

# Robot drives in a circle
angles = np.linspace(0, 2*np.pi, n_poses, endpoint=False)
radius = 5.0
true_poses = np.column_stack([radius*np.cos(angles), radius*np.sin(angles)])

# Build a simple pose graph
# State: all poses stacked [x0,y0, x1,y1, ..., xN,yN]
state_dim = 2 * n_poses
H_list = []; z_list = []; R_list = []

# Odometry constraints (consecutive poses)
for i in range(n_poses - 1):
    H = np.zeros((2, state_dim))
    H[0, 2*i] = -1; H[0, 2*(i+1)] = 1
    H[1, 2*i+1] = -1; H[1, 2*(i+1)+1] = 1
    z = true_poses[i+1] - true_poses[i] + np.random.normal(0, sigma_odom, 2)
    H_list.append(H); z_list.append(z); R_list.append(np.eye(2) * sigma_odom**2)

# Loop closure: pose N-1 to pose 0
if show_loop_closure:
    H = np.zeros((2, state_dim))
    H[0, 2*(n_poses-1)] = -1; H[0, 0] = 1
    H[1, 2*(n_poses-1)+1] = -1; H[1, 1] = 1
    z = true_poses[0] - true_poses[-1] + np.random.normal(0, sigma_loop, 2)
    H_list.append(H); z_list.append(z); R_list.append(np.eye(2) * sigma_loop**2)

# Anchor first pose (fix gauge)
H_anchor = np.zeros((2, state_dim))
H_anchor[0, 0] = 1; H_anchor[1, 1] = 1
H_list.append(H_anchor); z_list.append(true_poses[0]); R_list.append(np.eye(2) * 0.001)

# Solve: least squares (H^T R^{-1} H)^{-1} H^T R^{-1} z
H_full = np.vstack(H_list)
z_full = np.concatenate(z_list)
R_full = np.zeros((len(z_full), len(z_full)))
idx = 0
for R_i in R_list:
    n = R_i.shape[0]
    R_full[idx:idx+n, idx:idx+n] = R_i
    idx += n

W = np.linalg.inv(R_full)
A = H_full.T @ W @ H_full
b = H_full.T @ W @ z_full
x_est = np.linalg.solve(A, b)
P_est = np.linalg.inv(A)

est_poses = x_est.reshape(-1, 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, title in [(axes[0], f"Loop closure: {show_loop_closure}"), (axes[1], "Covariance magnitudes")]:
    pass

ax = axes[0]
ax.plot(true_poses[:, 0], true_poses[:, 1], 'k--', lw=1, alpha=0.5, label='ground truth')
ax.plot(est_poses[:, 0], est_poses[:, 1], 'steelblue', lw=2, marker='o', ms=5, label='estimated')
for i in range(n_poses):
    cov_i = P_est[2*i:2*i+2, 2*i:2*i+2]
    draw_cov_ellipse(ax, est_poses[i], cov_i, n_std=3,
                     fill=True, facecolor='steelblue', alpha=0.15, edgecolor='steelblue', lw=1)
if show_loop_closure:
    ax.annotate("loop closure!", xy=est_poses[0], xytext=(est_poses[0, 0]+1, est_poses[0, 1]+1.5),
                arrowprops=dict(arrowstyle='->', color='tomato', lw=2),
                fontsize=12, color='tomato', fontweight='bold')
ax.set_title(f"Pose graph (loop closure = {show_loop_closure})", fontsize=13)
ax.set_aspect('equal'); ax.legend()

ax = axes[1]
sigmas = [np.sqrt(np.trace(P_est[2*i:2*i+2, 2*i:2*i+2])) for i in range(n_poses)]
ax.plot(sigmas, 'steelblue', lw=2, marker='o', ms=5)
ax.set_xlabel("Pose index"); ax.set_ylabel("Position σ (m)")
ax.set_title("Per-pose uncertainty", fontsize=13)

plt.tight_layout()
plt.show()

**Key observations:**
- Without loop closure, uncertainty grows monotonically along the trajectory.
- With loop closure, uncertainty is **distributed** across all poses, resulting in a much tighter overall estimate.
- The correlations in the covariance matrix are what allow loop closure to propagate corrections.
- SLAM is fundamentally about **managing and exploiting these correlations**.

---

## Exercises

### Exercise 19.1: Build the joint state

Create a SLAM state vector with 3 poses and 2 landmarks (all 2D).
Initialize the covariance as a diagonal matrix. Print the state dimension and the covariance shape.

In [ ]:
# Your code here
# state = [x0, y0, x1, y1, x2, y2, l0x, l0y, l1x, l1y]
# P = np.eye(10) * some_initial_uncertainty

### Exercise 19.2: Observe correlations growing

Start with a diagonal covariance (no correlations). Simulate 10 observations of different
landmarks from the robot. After each observation, plot the absolute covariance matrix
as a heatmap. Watch the off-diagonal blocks fill in.

In [ ]:
# Your code here
# Use the Kalman update formula: K = P @ H.T @ inv(H @ P @ H.T + R)
# P = (I - K @ H) @ P

### Exercise 19.3: Loop closure impact

Create a pose graph with 10 poses in a circle. Solve once WITHOUT loop closure and once WITH.
Compare the per-pose uncertainty. How much does loop closure reduce the maximum uncertainty?

In [ ]:
# Your code here

### Exercise 19.4: Correlation visualization (challenge)

Build a SLAM system with 5 poses and 4 landmarks. After running all observations,
compute the **correlation matrix** (normalize covariance by diagonal elements).
Visualize as a heatmap. Which pairs of landmarks are most correlated? Why?

In [ ]:
# Your code here
# correlation[i,j] = P[i,j] / sqrt(P[i,i] * P[j,j])